## ANN - Vectores TF-IDF

Se convierte la columna `tokens` de `data/final/` en un vector TF-IDF normalizado (L2) por proceso

- Equivale al `TfidfVectorizer` + coseno de lab3.2, con `HashingTF` + `IDF` de Spark ML

- Al normalizar, el coseno es el producto punto

- Experimento: dimensiones del hashing (`NUM_FEATURES`)

### Imports

In [1]:
import os, socket, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 200)

### Utilidades

- Constantes

In [ ]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
final_dir = proyecto / "data" / "final"
out_dir = proyecto / "03_ANN" / "temp" / "a_vectores_tfidf"
out_dir.mkdir(parents=True, exist_ok=True)
vectores_dir = out_dir / "vectores"

- Funciones

Parámetros y funciones compartidas de `03_ANN/funciones.ipynb`

In [ ]:
ruta_funciones = str(proyecto / "03_ANN" / "funciones.ipynb")
%run $ruta_funciones

- Spark

In [3]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName("ann-vectores")
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))     # 4g (vs 2g en 05): firmas/vectores y joins por pares pesan más que canastas
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")                                   # ~231k filas: 24 tareas bastan, las 200 por defecto quedarían casi vacías
           .config("spark.sql.autoBroadcastJoinThreshold", "-1")                           # sin broadcast automático: los parquets descomprimidos saturaban el driver
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### Verificación del cluster

In [4]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()
print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 0
Cores totales : 2
Memoria driver: 3.0 GB
UI del job    : http://localhost:4040


---
### 1. Carga de data

In [5]:
assert final_dir.exists(), f"No se encuentra {final_dir}; corre 01_EDA_KDD/KDD/a_transformaciones.ipynb"
procesos = (spark.read.parquet(str(final_dir)).select(*COLS_ANN)
                 .where(F.col("anio").isin(ANIOS) & (F.size("tokens") > 0)))
N = procesos.count()
print(f"Procesos con tokens: {N:,}")
procesos.select(F.size("tokens").alias("n")).select(F.min("n"), F.expr("percentile_approx(n, 0.5)").alias("mediana"), F.max("n")).show()
procesos.select("ocid", "categoria", F.slice("tokens", 1, 8).alias("tokens (8 primeros)")).show(3, truncate=70)

Procesos con tokens: 231,171
+------+-------+------+
|min(n)|mediana|max(n)|
+------+-------+------+
|     1|     16|   478|
+------+-------+------+

+---------------------------+---------+----------------------------------------------------------------------+
|                       ocid|categoria|                                                   tokens (8 primeros)|
+---------------------------+---------+----------------------------------------------------------------------+
|ocds-dgv273-seacev3-1085359|    GOODS|[adquisicion, ladrillos, refractarios, manta, fibra, ceramica, refi...|
|ocds-dgv273-seacev3-1090383|    GOODS|[contratacion, bienes, adquisicion, aditivos, concreto, obra, mejor...|
|ocds-dgv273-seacev3-1092010| SERVICES|[contratacion, profesional, servicio, medicina, ocupacional, vigila...|
+---------------------------+---------+----------------------------------------------------------------------+
only showing top 3 rows



### 2. TF-IDF y escritura de vectores

Se escriben en `temp/a_vectores_tfidf/vectores`, particionados por `categoria`, como vector de Spark ML y como arrays `idx`/`val`

In [6]:
t0 = time.time()
vec, idf_model = ajustar_tfidf(procesos)
vec = vec.withColumn("arr", vector_a_arrays("vec")).select("*", "arr.idx", "arr.val").drop("arr")
vec = vec.where(F.size("idx") > 0)
vec.write.mode("overwrite").partitionBy("categoria").parquet(str(vectores_dir))
print(f"Vectores guardados en {time.time() - t0:,.0f} s")
vectores = spark.read.parquet(str(vectores_dir))

Vectores guardados en 44 s


#### a. Verificación

Filas = procesos, norma 1 y nº de dimensiones activas por vector

In [7]:
n_vec = vectores.count()
chk = (vectores.select(F.size("idx").alias("nnz"), F.aggregate("val", F.lit(0.0), lambda a, x: a + x * x).alias("norma2"))
                .select(F.min("nnz").alias("nnz_min"), F.expr("percentile_approx(nnz, 0.5)").alias("nnz_mediana"), F.max("nnz").alias("nnz_max"),
                        F.min("norma2").alias("norma2_min"), F.max("norma2").alias("norma2_max")).first())
print(f"Vectores: {n_vec:,} (procesos: {N:,}) | dims activas {chk['nnz_min']}–{chk['nnz_max']} (mediana {chk['nnz_mediana']}) | "
      f"norma² {chk['norma2_min']:.4f}–{chk['norma2_max']:.4f} | {len(list(vectores_dir.glob('categoria=*')))} particiones (categoria)")
assert abs(chk["norma2_min"] - 1) < 1e-6 and abs(chk["norma2_max"] - 1) < 1e-6
vectores.groupBy("categoria").count().show()
vectores.select("ocid", F.slice("idx", 1, 5).alias("idx (5)"), F.slice("val", 1, 5).alias("val (5)")).show(3, truncate=60)

Vectores: 231,171 (procesos: 231,171) | dims activas 1–453 (mediana 16) | norma² 1.0000–1.0000 | 3 particiones (categoria)
+---------+------+
|categoria| count|
+---------+------+
| SERVICES|100050|
|    GOODS|104002|
|    WORKS| 27119|
+---------+------+

+---------------------------+------------------------------+------------------------------------------------------------+
|                       ocid|                       idx (5)|                                                     val (5)|
+---------------------------+------------------------------+------------------------------------------------------------+
|ocds-dgv273-seacev3-1092010|    [10, 706, 771, 1848, 2044]|[0.2307835845185198, 0.2894377757731381, 0.35394587037999...|
|ocds-dgv273-seacev3-1092308|  [29, 1960, 3391, 3914, 4007]|[0.5100899480821687, 0.371526355978396, 0.599200216077542...|
|ocds-dgv273-seacev3-1092343|[1221, 1814, 2228, 2375, 2635]|[0.3898801852519337, 0.34212051906716967, 0.2826671174748...|
+----------

### 3. Coseno nativo vs. Spark ML

Muestra de pares; `coseno()` (arrays) vs. producto punto de los vectores ML

In [8]:
# ~500 vectores al azar → 3,000 pares bastan para comprobar la igualdad; tolerancia 1e-9 = error de punto flotante
m = vectores.sample(fraction=min(1.0, 500 / n_vec), seed=SEED).select("ocid", "vec", "idx", "val")
a, b = m.alias("a"), m.alias("b")
dot_ml = F.udf(lambda u, v: float(np.dot(u.toArray(), v.toArray())), "double")
chk2 = (a.join(b, F.col("a.ocid") < F.col("b.ocid")).limit(3000)
         .select(coseno(F.col("a.idx"), F.col("a.val"), F.col("b.idx"), F.col("b.val")).alias("nativo"), dot_ml(F.col("a.vec"), F.col("b.vec")).alias("ml"))
         .select(F.max(F.abs(F.col("nativo") - F.col("ml"))).alias("dif_max"), F.avg("nativo").alias("cos_promedio"), F.count("*").alias("pares")).first())
print(f"Pares: {chk2['pares']:,} | diferencia máxima nativo vs ML: {chk2['dif_max']:.2e} | coseno promedio entre procesos al azar: {chk2['cos_promedio']:.3f}")
assert chk2["dif_max"] < 1e-9

Pares: 3,000 | diferencia máxima nativo vs ML: 5.55e-17 | coseno promedio entre procesos al azar: 0.015


- 231,171 vectores en 44 s, con mediana de 16 dimensiones activas y norma 1

- Diferencia máxima entre el coseno nativo y el de Spark ML: 5.5e-17 (error de punto flotante)

- Coseno promedio entre procesos al azar: 0.015 — dos descripciones cualesquiera casi no comparten palabras, así que un coseno alto sí indica objetos parecidos

### 4. Experimento: dimensiones del hashing (`NUM_FEATURES`)

- `HashingTF` manda cada token a una de `NUM_FEATURES` dimensiones; con pocas dimensiones, tokens distintos chocan y el coseno se distorsiona

- Referencia sin colisiones: `CountVectorizer` (cada token su propia dimensión) sobre una muestra de ~20k procesos

- Se compara el coseno de pares de la misma entidad con 1,024, 4,096 y 16,384 dimensiones

In [ ]:
from pyspark.ml.feature import CountVectorizer

EXP_FEATURES = [1024, 4096, 16384]   # 2¹⁰, 2¹² (actual) y 2¹⁴: un cuarto, igual y 4 veces las dimensiones
muestra_nf = procesos.sample(fraction=min(1.0, 20_000 / N), seed=SEED).select("ocid", "entidad", "tokens").cache()   # ~20k: corre en segundos
n_nf = muestra_nf.count()

# referencia: vocabulario exacto, sin hashing
cv = CountVectorizer(inputCol="tokens", outputCol="tf", minDF=MIN_DF).fit(muestra_nf)
tf_ex = cv.transform(muestra_nf)
ref = Normalizer(inputCol="tfidf", outputCol="vec", p=2.0).transform(IDF(inputCol="tf", outputCol="tfidf").fit(tf_ex).transform(tf_ex))
V = len(cv.vocabulary)

# pares de la misma entidad: tienen cosenos altos y medios, no solo ~0 como los pares al azar
pares_nf = (muestra_nf.alias("a").join(muestra_nf.alias("b"), (F.col("a.entidad") == F.col("b.entidad")) & (F.col("a.ocid") < F.col("b.ocid")))
                      .select(F.col("a.ocid").alias("o1"), F.col("b.ocid").alias("o2")).limit(5000).cache())
dot = F.udf(lambda u, v: float(u.dot(v)), "double")

def cosenos(df_vec, nombre):
    v1 = df_vec.select(F.col("ocid").alias("o1"), F.col("vec").alias("v1"))
    v2 = df_vec.select(F.col("ocid").alias("o2"), F.col("vec").alias("v2"))
    return pares_nf.join(v1, "o1").join(v2, "o2").select("o1", "o2", dot("v1", "v2").alias(nombre))

cos_ref = cosenos(ref, "cos_exacto").cache()
print(f"Muestra: {n_nf:,} procesos | vocabulario exacto: {V:,} tokens (df ≥ {MIN_DF}) | pares: {cos_ref.count():,}")

filas = []
for nf in EXP_FEATURES:
    t0 = time.time()
    vec_nf, _ = ajustar_tfidf(muestra_nf, num_features=nf)
    c = (cos_ref.join(cosenos(vec_nf, "cos_hash"), ["o1", "o2"])
                .withColumn("err", F.abs(F.col("cos_hash") - F.col("cos_exacto"))))
    r_ = c.agg(F.avg("err").alias("mae"), F.max("err").alias("err_max"), F.avg((F.col("err") > 0.05).cast("int")).alias("frac")).first()
    filas.append({"NUM_FEATURES": nf, "tokens_por_dim": round(V / nf, 2), "MAE_coseno": round(r_["mae"], 4),
                  "error_max": round(r_["err_max"], 4), "frac_err>0.05": round(r_["frac"], 3), "seg": round(time.time() - t0, 1)})
tabla_nf = pd.DataFrame(filas)
tabla_nf

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
x = tabla_nf["NUM_FEATURES"].astype(str)
ax[0].bar(x, tabla_nf["MAE_coseno"], color="#2b6cb0"); ax[0].set_title("Error del coseno vs. vocabulario exacto"); ax[0].set_xlabel("NUM_FEATURES")
ax[1].bar(x, tabla_nf["frac_err>0.05"] * 100, color="#2b6cb0"); ax[1].set_title("% de pares con error > 0.05"); ax[1].set_xlabel("NUM_FEATURES")
plt.tight_layout(); plt.show()

- Qué mirar: con más tokens que dimensiones (`tokens_por_dim` > 1) las colisiones son seguras; el error debería bajar al subir `NUM_FEATURES`

- El costo de subirlo está en el índice IVF: cada centroide es denso, así que 16,384 dimensiones = 4 veces la memoria de los 300 centroides

---
### 5. Cerrar sesión

In [9]:
spark.stop()